In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "torch", "matplotlib"])

if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
import numpy as np

from evaluation import (
    alda_advise,
    average_seeds,
    discover_runs,
    format_alda_report,
    format_metric_table,
    format_palm_report,
    load_curves,
    palm_evaluate,
)
from evaluation.plots import plot_accuracy_curves

In [ ]:
RESULTS_ROOT = "/kaggle/input/EDIT_RUN_OUTPUT_SLUG"

DATASET = "histoset"  # pathmnist | histoset | skintissue
SEEDS = None  # None = every seed found | e.g. [42, 38, 102]

METRICS = ["acc", "precision", "recall", "f1"]  # any subset, in report order
LABEL_BY = "method_label"  # method_label | run_name | sampler

INCLUDE_FINAL_TRAINING = False  # True mixes LoRA/aux/augment runs into the table

TARGET = None  # None = 95% of the best observed accuracy | e.g. 0.80
DELTA_TARGET = None  # None = 0.05 for fractional scores | e.g. 0.03
ETA = 0.05  # tolerated relative B_abs increase inside C_eta
PILOT_POINTS = 4  # budgets ALDA may use in the pilot rerun; None disables it
ALDA_SEED = 0  # seeds the curve fit's random restarts

HIGHLIGHT_PREFIX = "pact"  # method drawn thicker in the figure

In [ ]:
runs = discover_runs(
    RESULTS_ROOT,
    dataset=DATASET,
    seeds=SEEDS,
    include_final_training=INCLUDE_FINAL_TRAINING,
)
assert runs, f"no runs found for dataset={DATASET!r} under {RESULTS_ROOT}"

budgets = sorted({budget for run in runs for budget in run.budgets})
seeds_found = sorted({run.seed for run in runs})
print(f"{len(runs)} runs | dataset={DATASET} | seeds={seeds_found} | budgets={budgets}")
for run in sorted(runs, key=lambda item: (item.method_label, item.seed)):
    print(f"  {run.method_label:<34} seed={run.seed:<5} budgets={len(run.budgets)}")

In [ ]:
for metric in METRICS:
    print(format_metric_table(runs, metric=metric, label_by=LABEL_BY))
    print()

In [ ]:
curves = load_curves(runs, metric="acc", label_by=LABEL_BY)
averaged = average_seeds(curves, budgets)

seed_counts = sorted({entry["n_seeds"] for entry in averaged.values()})
if seed_counts != [len(seeds_found)]:
    print(f"[warning] methods do not all have the same seed count: "
          f"{ {m: averaged[m]['n_seeds'] for m in sorted(averaged)} }")
    print("          PALM and ALDA below are fitted on curves averaged over "
          "different numbers of seeds, so they are not equally noisy.")

print(f"{'method':<34}{'budget':>8}{'mean':>9}{'std':>9}{'seeds':>7}")
for method in sorted(averaged):
    entry = averaged[method]
    for budget, mean, std in zip(entry["budgets"], entry["accuracies"], entry["std"]):
        print(f"{method:<34}{budget:>8}{mean:>9.4f}{std:>9.4f}{entry['n_seeds']:>7}")

In [ ]:
palm_rows = {}
for method in sorted(averaged):
    entry = averaged[method]
    try:
        palm_rows[method] = palm_evaluate(entry["budgets"], entry["accuracies"])
    except (RuntimeError, ValueError) as error:
        print(f"[PALM] {method}: {error}")

header = (f"{'method':<34}{'Amax':>8}{'delta':>9}{'alpha':>9}{'beta':>8}"
          f"{'AUC':>8}{'B->90%':>9}{'RMSE':>8}")
print(header)
print("-" * len(header))
for method, params in sorted(palm_rows.items(), key=lambda kv: -kv[1]["auc_normalized"]):
    if not params["fit_success"]:
        print(f"{method:<34}  fit did not converge")
        continue
    target_budget = params["budget_to_90pct_amax"]
    print(f"{method:<34}{params['Amax']:>8.4f}{params['delta']:>9.4f}"
          f"{params['alpha']:>9.3f}{params['beta']:>8.3f}"
          f"{params['auc_normalized']:>8.4f}"
          f"{('-' if target_budget is None else format(target_budget, '.0f')):>9}"
          f"{params['fit_rmse']:>8.4f}")

In [ ]:
for method in sorted(palm_rows):
    print(format_palm_report(palm_rows[method], method, DATASET))

In [ ]:
best_observed = max(max(entry["accuracies"]) for entry in averaged.values())
target = TARGET if TARGET is not None else round(0.95 * best_observed, 3)
if TARGET is None:
    print(f"target not set: using 95% of the best observed accuracy "
          f"({best_observed:.4f}) -> {target}")

advice = alda_advise(
    averaged,
    target=target,
    delta_target=DELTA_TARGET,
    eta=ETA,
    seed=ALDA_SEED,
)
print(format_alda_report(advice, DATASET))

In [ ]:
if PILOT_POINTS is not None and PILOT_POINTS < len(budgets):
    pilot = alda_advise(
        averaged,
        target=target,
        delta_target=DELTA_TARGET,
        eta=ETA,
        max_points=PILOT_POINTS,
        seed=ALDA_SEED,
    )
    print(format_alda_report(pilot, DATASET))

    chosen_full = advice["advice"]["selected_method"]
    chosen_pilot = pilot["advice"]["selected_method"]
    print(f"full-curve pick : {chosen_full}")
    print(f"pilot pick      : {chosen_pilot} "
          f"(first {PILOT_POINTS} budgets, up to {budgets[PILOT_POINTS - 1]} labels)")
    print("pilot agrees with the full curve"
          if chosen_full == chosen_pilot else
          "PILOT DISAGREES: the short pilot would have committed to a different method")
else:
    print(f"pilot rerun skipped (PILOT_POINTS={PILOT_POINTS}, budgets={len(budgets)})")

In [ ]:
highlight = next(
    (method for method in sorted(averaged) if method.startswith(HIGHLIGHT_PREFIX)), None
)
if highlight is None:
    print(f"[figure] no method starts with {HIGHLIGHT_PREFIX!r}: nothing highlighted")

plot_accuracy_curves(
    budgets_dict={DATASET: budgets},
    acc_data=[{method: [value * 100.0 for value in averaged[method]["accuracies"]]
               for method in sorted(averaged)}],
    highlight=highlight,
    dataset_titles={DATASET: DATASET},
    save_path=f"/kaggle/working/accuracy_{DATASET}.png",
)